# **Client Project**

# **Project Name : Intelligent RAG-Based Chatbot Using LangChain & LLMs**


# Project Objective

The objective of this project is to develop an intelligent Retrieval-Augmented Generation (RAG) chatbot that can answer user queries using domain-specific documents.

The chatbot retrieves relevant information from the knowledge base and provides grounded responses using a Large Language Model (LLM).

The project implements the complete RAG pipeline including document ingestion, text chunking, embeddings, vector store creation, information retrieval, prompt augmentation, response generation, evaluation, and chatbot interaction.

# Introduction

Large Language Models can generate natural and informative responses, but they may produce incorrect or hallucinated information when asked about specific or domain-dependent topics.

Retrieval-Augmented Generation (RAG) addresses this problem by retrieving relevant information from an external knowledge base and providing it as context to the LLM before generating the final response.

In this project, a RAG-based chatbot is developed using LangChain, embeddings, FAISS, and an LLM. The system retrieves relevant document chunks and uses them to generate context-aware and grounded responses.

#  Technologies Used

- Python
- Google Colab
- LangChain
- Groq LLM
- Hugging Face Sentence Transformers
- FAISS Vector Database
- PyPDF
- Gradio
- RAGAS
- Datasets

## Main Components

1. LangChain – RAG pipeline orchestration
2. PyPDF – PDF document loading
3. Sentence Transformers – text embeddings
4. FAISS – vector similarity search
5. Groq – Large Language Model inference
6. Gradio – chatbot user interface
7. RAGAS – RAG system evaluation

# Problem Statement

Generic LLMs may not have access to the latest or domain-specific information contained in private or external documents.

The objective is to develop a chatbot that can retrieve relevant information from a given document collection and use an LLM to answer user queries accurately based on the retrieved context.

# 1. Install Libraries

In [ ]:
!pip install -q langchain-huggingface sentence-transformers
!pip install -q faiss-cpu pypdf
!pip install -q gradio ragas datasets
!pip install -q langchain-groq

# 2. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd

from google.colab import files, userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate

print("Libraries imported successfully!")

# 3. Upload Knowledge Base

In [ ]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_file)

# Knowledge Base Used

A custom Artificial Intelligence and Machine Learning knowledge base was selected as the document source for this RAG chatbot.

The PDF provides a controlled and self-contained knowledge source, making it suitable for demonstrating document-based Retrieval-Augmented Generation.

## Why PDF?

PDF was selected because:

1. It provides a structured and self-contained knowledge source.
2. It is easy to process using LangChain and PyPDF.
3. It supports reproducible document-based question answering.
4. It avoids dependency on continuously changing external websites.
5. It contains multiple AI/ML topics, allowing retrieval to be tested across different concepts.

## Topics Covered

The knowledge base contains the following topics:

- Artificial Intelligence
- Machine Learning
- Supervised, Unsupervised and Reinforcement Learning
- Data Preprocessing
- Machine Learning Algorithms and Evaluation Metrics
- Deep Learning and Neural Networks
- Natural Language Processing
- Computer Vision
- Generative AI and Large Language Models
- Retrieval-Augmented Generation (RAG)
- AI Chatbots
- AI Tools, Libraries and Programming
- Real-World Applications of AI
- Responsible AI and Ethics
- Future Scope of AI
- AI/ML Concept Comparisons
- AI/ML Glossary

# 3.1 Load PDF

In [ ]:
loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

# 4. Check PDF Content

In [ ]:
print(documents[0].page_content[:2001])

# 5. Split PDF Into Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("PDF split into chunks successfully!")
print("Number of chunks:", len(chunks))

# Check a Chunk

In [ ]:
print(chunks[0].page_content)

# 6. Create Embeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

# 7. Create FAISS Vector Store

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully!")

#  Save Vector Store

In [ ]:
vectorstore.save_local("faiss_index")

print("FAISS vector store saved successfully!")

# 8. Create Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

#  Test Retrieval

In [ ]:
query = "What is machine learning?"

retrieved_docs = retriever.invoke(query)

print("Retrieved Documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print("\n--- Document", i + 1, "---")
    print(doc.page_content[:500])

# 9. Add Groq API Key

In [ ]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded successfully!")
except userdata.SecretNotFoundError:
    print("ERROR: GROQ_API_KEY secret not found!")
    print("Please click the key icon (Secrets) on the left sidebar, add a new secret named 'GROQ_API_KEY', paste your API key, and enable Notebook Access.")

# 10. Initialize Groq LLM

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=GROQ_API_KEY,
    temperature=0
)

print("Groq LLM initialized successfully!")

# 11. Create RAG Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful question-answering assistant.

Answer the user's question using only the provided context.

If the answer is not available in the context, clearly say:
"I could not find the answer in the provided document."

Do not make up information.

Context:
{context}

Question:
{question}

Answer:
""")

print("RAG prompt created successfully!")

# 12. Create RAG Function

In [ ]:
def rag_answer(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)

    return response.content

# 13. Test Question

In [ ]:
question = "What Deep learning?"

try:
    answer = rag_answer(question)

    print("Question:", question)
    print("\nAnswer:", answer)

except Exception as e:
    print("An error occurred:", e)

# 14. Test Multiple Questions

In [ ]:
questions = [
    "What is machine learning?",
    "What are the main concepts discussed in the document?",
    "What are the applications mentioned?",
    "What is Computer Vision?"

]

for question in questions:

    print("\nQuestion:", question)

    answer = rag_answer(question)

    print("Answer:", answer)
    print("-" * 80)

# 17. Basic Answer Evaluation

In [ ]:
evaluation_questions = [
    "What is Artificial Intelligence?",
    "What are the types of Machine Learning?",
    "Why RAG is useful?",
    "How does a neural network learn during training?",
    "What is Natural Language Processing?",
    "What is Computer Vision?",
    "What is Generative AI?",
    "What is Retrieval-Augmented Generation?",
    "How does a machine learning model predict the price of a new house?",
    "What are the limitations of AI?"
]

evaluation_results = []

for question in evaluation_questions:

    answer = rag_answer(question)

    evaluation_results.append({
        "Question": question,
        "Answer": answer
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

# 18. Retrieve Context for Evaluation

In [ ]:
def retrieve_context(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    return context

# 19. Generate Evaluation Dataset

In [ ]:
evaluation_data = []

for question in evaluation_questions:

    context = retrieve_context(question)
    answer = rag_answer(question)

    evaluation_data.append({
        "question": question,
        "contexts": [context],
        "answer": answer
    })

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

# 20. Display Retrieved Context

In [ ]:
for i, row in evaluation_df.iterrows():

    print(f"\n{'='*80}")
    print(f"Question {i+1}: {row['question']}")

    print("\nRetrieved Context:")
    print(row["contexts"][0][:1000])

    print("\nGenerated Answer:")
    print(row["answer"])

# Create References Answers

In [ ]:
reference_answers = [
    "Artificial Intelligence (AI) is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include understanding information, recognizing patterns, making decisions, solving problems, understanding language, and learning from experience.",

    "The main types of Machine Learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labelled data, unsupervised learning uses unlabelled data to find hidden structures, and reinforcement learning learns through rewards and penalties while interacting with an environment.",

    "RAG is useful because it grounds answers in real documents, reduces hallucination, allows private or up-to-date information to be used without retraining the model, is easier to update than fine-tuning, and can provide the source of the answer.",

    "A neural network learns during training by processing data and adjusting its parameters based on the error in its predictions. The training process uses a loss function to measure how wrong the predictions are and updates the model so that its predictions improve.",

    "Natural Language Processing (NLP) is a branch of Artificial Intelligence that enables computers to work with and understand human language.",

    "Computer Vision is a branch of Artificial Intelligence that enables systems to process and understand visual information such as images and videos.",

    "Generative AI refers to AI systems that can generate new content such as text, images, audio, video, and code. Large Language Models are an important example of Generative AI systems.",

    "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with a Large Language Model. It retrieves relevant information from an external knowledge source and uses that information to generate a grounded answer.",

    "A machine learning model predicts the price of a new house by learning patterns from historical housing data. For a regression problem such as house-price prediction, the model learns the relationship between features such as property characteristics and the known house prices, and then uses those learned patterns to predict the price of a new house.",

    "AI and Machine Learning have limitations such as dependence on data quality, bias in training data, difficulty interpreting complex models, high computational requirements, generalization problems, hallucinations in Generative AI, and privacy and security concerns."
]

print("Reference answers created successfully!")
print("Number of reference answers:", len(reference_answers))

# 22. RAGAS Evaluation

# 22.1 Check RAGAS Version

In [ ]:
import importlib.metadata

ragas_version = importlib.metadata.version("ragas")

print("RAGAS version:", ragas_version)

# 22.2 Prepare Dataset for RAGAS Evaluation

In [ ]:
evaluation_df["reference"] = reference_answers

evaluation_df

# 22.3 RAGAS Evaluation - Faithfulness & Relevancy

In [ ]:
import sys
from unittest.mock import MagicMock

# Mock the missing legacy VertexAI modules to prevent Ragas import from crashing
sys.modules["langchain_community.chat_models.vertexai"] = MagicMock()
sys.modules["langchain_community.llms.vertexai"] = MagicMock()

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset

# Prepare the dataset
ragas_dataset = Dataset.from_pandas(
    evaluation_df[
        ["question", "contexts", "answer", "reference"]
    ]
)

# Run the evaluation explicitly passing your Groq LLM and Hugging Face Embeddings
ragas_result = evaluate(
    ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=llm,
    embeddings=embeddings
)

ragas_result

# 22.4 Display RAGAS Score

In [ ]:
ragas_scores = ragas_result.to_pandas()

ragas_scores

In [ ]:
faithfulness_score = ragas_scores["faithfulness"].mean()
relevancy_score = ragas_scores["answer_relevancy"].mean()

print("Average Faithfulness Score:", round(faithfulness_score, 4))
print("Average Answer Relevancy Score:", round(relevancy_score, 4))

# 22.5 Basic Evaluation Summary

In [ ]:
evaluation_summary = evaluation_df[
    ["question", "answer"]
].copy()

evaluation_summary["Answer Available"] = evaluation_summary["answer"].apply(
    lambda x: "Yes"
    if "could not find the answer" not in x.lower()
    else "No"
)

evaluation_summary

# 23. Retrieval Quality Check

In [ ]:
retrieval_results = []

for question in evaluation_questions:

    retrieved_docs = retriever.invoke(question)

    retrieval_results.append({
        "Question": question,
        "Retrieved Chunks": len(retrieved_docs)
    })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df

### This verifies that the retriever is consistently returning documents for the evaluation questions.

# 24. Add Source / Page Information

In [ ]:
def get_sources(question):

    retrieved_docs = retriever.invoke(question)

    sources = []

    for doc in retrieved_docs:

        page = doc.metadata.get("page", None)

        if page is not None:
            sources.append(f"Page {page + 1}")
        else:
            sources.append("Unknown page")

    return list(dict.fromkeys(sources))

### Because PyPDFLoader stores page metadata, we can identify which PDF pages contributed to the answer.

# 25. Test Sources

In [ ]:
question = "What is machine learning?"

print("Question:", question)
print("Sources:", get_sources(question))

# 26. Create a Professional RAG Response

In [ ]:
def rag_answer_with_sources(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)

    sources = []

    for doc in retrieved_docs:

        page = doc.metadata.get("page")

        if page is not None:
            sources.append(f"Page {page + 1}")

    sources = list(dict.fromkeys(sources))

    return response.content, sources

In [ ]:
from langchain_groq import ChatGroq

# Re-initialize the LLM using an active, supported model
llm = ChatGroq(
       model="openai/gpt-oss-20b",

    groq_api_key=GROQ_API_KEY,
    temperature=0,
    n=1
)

question = "What is Artificial Intelligence?"

answer, sources = rag_answer_with_sources(question)

print("Answer:")
print(answer)

print("\nSources:")
print(", ".join(sources))

# 27.  Gradio Chatbot With Sources

In [ ]:

!pip install -q --upgrade gradio uvicorn

In [ ]:

import asyncio
import asyncio.runners

if "_patch_asyncio" in getattr(asyncio.run, "__qualname__", ""):
    asyncio.run = asyncio.runners.run

print("Asyncio compatibility check completed.")
print("Current asyncio.run:", asyncio.run)


In [ ]:

import asyncio
import asyncio.runners

# Restore the original asyncio.run function
asyncio.run = asyncio.runners.run

print("asyncio.run restored:", asyncio.run is asyncio.runners.run)


In [ ]:

import gradio as gr

def chatbot_response(message, history):
    try:
        answer, sources = rag_answer_with_sources(message)

        if sources:
            answer += "\n\n**Sources:** " + ", ".join(sources)

        return answer

    except Exception as e:
        return f"Error: {type(e).__name__}: {str(e)}"

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="AI & ML RAG Chatbot",
    description="Ask questions about Artificial Intelligence and Machine Learning.",
    examples=[
        "What is Artificial Intelligence?",
        "What is Machine Learning?",
        "What is Deep Learning?"
    ]
)

demo.launch(share=False, debug=True)


# Enhanced RAG Chatbot with Source References

The chatbot is enhanced to display the source pages from the knowledge base along with the generated answer.

This feature improves transparency and allows users to verify the information by referring to the original document. The enhanced chatbot is used as the final chatbot interface for this project.


#  Limitations

- The chatbot's document-grounded responses depend on the quality and coverage of the knowledge base.
- If relevant information is not available in the knowledge base, the chatbot may not be able to provide a document-grounded answer.
- Retrieval quality depends on chunk size, chunk overlap, embedding quality, and the number of retrieved chunks.
- The system does not automatically update its knowledge base when new information becomes available.
- The quality of generated responses also depends on the selected LLM.
- The current knowledge base is limited to the selected Artificial Intelligence and Machine Learning document.

#  Future Scope

The project can be extended in several ways:

1. Support multiple PDF documents.
2. Add website and web-page ingestion.
3. Add CSV and structured-data retrieval.
4. Use advanced embedding models.
5. Experiment with different vector databases.
6. Add conversation memory.
7. Add authentication and user management.
8. Add document upload directly through the chatbot interface.
9. Implement automatic knowledge-base updating.
10. Improve retrieval using hybrid search or reranking.
11. Deploy the chatbot as a web application.
12. Add advanced RAG evaluation and monitoring.

#  Conclusion

The Intelligent RAG-Based Chatbot project successfully demonstrates the integration of document retrieval, vector embeddings, semantic search, and Large Language Model-based response generation.

A custom Artificial Intelligence and Machine Learning knowledge base was processed using LangChain and PyPDF. The document was divided into meaningful chunks and converted into vector embeddings using a Hugging Face Sentence Transformer model.

The generated embeddings were stored in a FAISS vector database, enabling the system to retrieve relevant information for user queries.

A Groq-based Large Language Model was then integrated into the RAG pipeline to generate responses using the retrieved context.

The project also includes a Gradio-based chatbot interface, sample question testing, source retrieval, and RAG evaluation components.

Overall, the project demonstrates how Retrieval-Augmented Generation can be used to build a domain-specific chatbot that combines external knowledge retrieval with the language-generation capabilities of an LLM.